# 01 — Upper Bound Back-test: Sequential FCR-D Bidding (Perfect Forecasts)

This notebook computes the **upper bound** on revenue for a simulated 1 MW BTM battery portfolio
participating in the FCR-D up and down markets (DK2) via sequential day-ahead auctions (early + late).

**All forecasts are perfect** — actual realised values are fed directly into the MILP.
No noise, no randomness in predictions. This gives the theoretical maximum revenue.

Noise / sensitivity analysis belongs in `02_Sensitivity.ipynb`.

### Model summary
- Two MILPs per day: **Early auction** (00:00 DA) and **Late auction** (17:00 DA)
- Objectives: minimise net electricity cost while maximising FCR-D reservation revenue
- Competing trade-offs: self-consumption + arbitrage vs. ancillary services capacity reservation
- Solver: HiGHS via Pyomo `appsi_highs`

### File structure expected
```
project/
├── data/
│   ├── data_in/
│   │   ├── data_ec/       # EC base profiles (b-type, s-type)
│   │   ├── data_FCRD/     # FCR-D price data
│   │   ├── data_freq/     # Frequency / activation fraction data
│   │   └── data_spot/     # Spot price data
│   └── data_out/
│       └── combined_2025_with_frequency.csv
├── notebooks/
│   ├── 01_Back_test.ipynb  ← this file
│   └── 02_Sensitivity.ipynb
└── figures/
```

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings, os, time
from datetime import datetime, timedelta

import pyomo.environ as pyo
from pyomo.contrib.appsi.solvers.highs import Highs

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.max_open_warning': 0, 'figure.dpi': 120})

print('All imports OK')

All imports OK


## 1. Configuration & Parameters (Table 2)

In [10]:
# ── Paths (adjust to your layout) ─────────────────────────────────────────
# When running from notebooks/ folder:
BASE_DIR       = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_IN_DIR    = os.path.join(BASE_DIR, 'data', 'data_in')
DATA_OUT_DIR   = os.path.join(BASE_DIR, 'data', 'data_out')
FIGURES_DIR    = os.path.join(BASE_DIR, 'figures')
EC_DIR         = os.path.join(DATA_IN_DIR, 'data_ec')

COMBINED_CSV   = os.path.join(DATA_OUT_DIR, 'combined_2025_with_frequency.csv')

os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Portfolio parameters (Table 2) ────────────────────────────────────────
N_EC           = 10        # number of energy communities
MAX_POWER_EC   = 100       # kW per EC battery
CAPACITY_EC    = 200       # kWh per EC battery
ETA            = 0.95      # one-way efficiency; round-trip = eta^2
SOC_INIT_FRAC  = 0.50      # initial & terminal SOC fraction
P_MIN_BID      = 100       # kW minimum FCR-D bid in DK2
T_SUSTAIN      = 0.5       # hours sustain duration
T_MAX          = 24        # hours per day

# Derived
P_MAX_PORTFOLIO = N_EC * MAX_POWER_EC   # 1000 kW
S_MAX_PORTFOLIO = N_EC * CAPACITY_EC     # 2000 kWh

# ── Synthetic portfolio generation ────────────────────────────────────────
# Deterministic seed — no randomness in upper bound
RNG_SEED       = 42
PROB_B_TYPE    = 0.60
PROB_S_TYPE    = 0.40
SCALE_MU       = 1.0
SCALE_SIGMA    = 1.0

print(f'Portfolio: {N_EC} ECs × {MAX_POWER_EC} kW / {CAPACITY_EC} kWh = '
      f'{P_MAX_PORTFOLIO} kW / {S_MAX_PORTFOLIO} kWh')

Portfolio: 10 ECs × 100 kW / 200 kWh = 1000 kW / 2000 kWh


In [14]:
df_all = pd.read_csv(COMBINED_CSV)
print(list(df_all.columns))
print(df_all.head(3))

['hour_utc', 'ec_id', 'unique_id', 'consumption', 'pv_production_kwh', 'spot_exkl_vat_ore_kwh', 'buy_price_inkl_vat_ore_kwh', 'sell_price_inkl_vat_ore_kwh', 'price_ore_kwh_fcr_d_ned__d_1_early', 'price_ore_kwh_fcr_d_ned__d_1_late', 'price_ore_kwh_fcr_d_ned__total', 'price_ore_kwh_fcr_d_upp__d_1_early', 'price_ore_kwh_fcr_d_upp__d_1_late', 'price_ore_kwh_fcr_d_upp__total', 'price_ore_kwh_fcr_n__d_1_early', 'price_ore_kwh_fcr_n__d_1_late', 'price_ore_kwh_fcr_n__total', 'mean_temp', 'mean_radiation', 'mean_wind_speed', 'mean_relative_hum', 'acc_precip', 'temp_forecast', 'solar_forecast', 'wind_forecast', 'relative_hum_forecast', 'precip_forecast', 'timestamp_utc', 'freq_avg_hz', 'y_act_fcrd_up', 'y_act_fcrd_down', 'y_act_fcrn_up', 'y_act_fcrn_down', 'data_quality_flag']
                    hour_utc ec_id         unique_id  consumption  \
0  2025-01-01 00:00:00+00:00     b  dk2_community_01          0.0   
1  2025-01-01 00:00:00+00:00     s  dk2_community_01          0.0   
2  2025-01-01 0

## 2. Load combined market data

In [13]:
# ── Load the pre-processed combined CSV ──────────────────────────────────
df_all = pd.read_csv(COMBINED_CSV)
df_all = df_all.sort_values('HourUTC').reset_index(drop=True)

print(f'Loaded {len(df_all)} rows, columns:')
print(list(df_all.columns))
df_all.head(3)

KeyError: 'HourUTC'

In [12]:
# ══════════════════════════════════════════════════════════════════════════
# COLUMN MAPPING — adjust these to match your CSV column names
# ══════════════════════════════════════════════════════════════════════════
COL_HOUR              = 'HourUTC'
COL_SPOT              = 'SpotPriceDKK'           # DKK/kWh
COL_TARIFF_IMPORT     = 'TariffImport'           # DKK/kWh added on top of spot for buying
COL_TARIFF_EXPORT     = 'TariffExport'           # DKK/kWh subtracted from spot for selling
COL_FCRD_UP_EARLY     = 'FCRD_Up_Early_DKK_kWh'   # DKK/kW/h
COL_FCRD_DOWN_EARLY   = 'FCRD_Down_Early_DKK_kWh'
COL_FCRD_UP_LATE      = 'FCRD_Up_Late_DKK_kWh'
COL_FCRD_DOWN_LATE    = 'FCRD_Down_Late_DKK_kWh'
COL_ACT_FRAC_UP       = 'ActFrac_Up'             # fraction [0,1]
COL_ACT_FRAC_DOWN     = 'ActFrac_Down'            # fraction [0,1]

# ── Derive buy / sell prices ──────────────────────────────────────────────
# If your CSV already has buy/sell price columns, map them here instead.
# Buy price  = spot + tariff (incl. VAT)
# Sell price  = spot - tariff (export)
if COL_TARIFF_IMPORT in df_all.columns:
    df_all['buy_price']  = df_all[COL_SPOT] + df_all[COL_TARIFF_IMPORT]
    df_all['sell_price'] = df_all[COL_SPOT] - df_all.get(COL_TARIFF_EXPORT, 0)
else:
    # Fallback: check if buy/sell columns already exist
    for cand_buy in ['buy_price', 'BuyPrice', 'import_price', 'ImportPrice']:
        if cand_buy in df_all.columns:
            df_all['buy_price'] = df_all[cand_buy]
            break
    for cand_sell in ['sell_price', 'SellPrice', 'export_price', 'ExportPrice']:
        if cand_sell in df_all.columns:
            df_all['sell_price'] = df_all[cand_sell]
            break
    if 'buy_price' not in df_all.columns:
        raise KeyError('Cannot derive buy/sell prices. Check COL_TARIFF_IMPORT or add buy_price column.')

# ── Ensure FCR-D price columns exist (fill 0 if missing) ─────────────────
for col in [COL_FCRD_UP_EARLY, COL_FCRD_DOWN_EARLY,
            COL_FCRD_UP_LATE, COL_FCRD_DOWN_LATE,
            COL_ACT_FRAC_UP, COL_ACT_FRAC_DOWN]:
    if col not in df_all.columns:
        print(f'WARNING: column {col} not found, filling with 0')
        df_all[col] = 0.0

# ── Buy-back price = max(early, late) for each direction ──────────────────
df_all['buyback_up']   = df_all[[COL_FCRD_UP_EARLY,   COL_FCRD_UP_LATE]].max(axis=1)
df_all['buyback_down'] = df_all[[COL_FCRD_DOWN_EARLY, COL_FCRD_DOWN_LATE]].max(axis=1)

# ── Acceptance flag: assume accepted wherever price > 0 (Assumption A1) ──
df_all['y_acc_up']   = (df_all[COL_FCRD_UP_EARLY]   > 0).astype(float)
df_all['y_acc_down'] = (df_all[COL_FCRD_DOWN_EARLY] > 0).astype(float)

# Add date column for grouping
df_all['date'] = df_all[COL_HOUR].dt.date

print(f'Price columns OK. Date range: {df_all[COL_HOUR].min()} → {df_all[COL_HOUR].max()}')

KeyError: 'Cannot derive buy/sell prices. Check COL_TARIFF_IMPORT or add buy_price column.'

## 3. Load / generate EC profiles (deterministic)

In [ ]:
def load_ec_base_profiles(ec_dir):
    """
    Try to load b-type and s-type base profiles from data_ec/.
    Expected: CSV files with hourly Load and PV columns for a reference day.
    Returns dict with keys 'b' and 's', each having 'load' and 'pv' arrays (24,).
    """
    profiles = {}
    for ptype in ['b', 's']:
        fname = os.path.join(ec_dir, f'{ptype}_type_profile.csv')
        if os.path.exists(fname):
            tmp = pd.read_csv(fname)
            profiles[ptype] = {
                'load': tmp['Load'].values[:24],
                'pv':   tmp['PV'].values[:24]
            }
    return profiles if len(profiles) == 2 else None


def load_ec_daily_profiles(ec_dir, date_val):
    """
    Try to load per-day, per-EC profiles.
    Expected: individual CSVs or a single large file with EC index + date.
    Returns: load_array (N_EC, 24), pv_array (N_EC, 24)
    """
    # Attempt: single file with all ECs and dates
    for fname in ['ec_profiles.csv', 'ec_load_pv.csv']:
        fpath = os.path.join(ec_dir, fname)
        if os.path.exists(fpath):
            tmp = pd.read_csv(fpath, parse_dates=['HourUTC'])
            day_data = tmp[tmp['HourUTC'].dt.date == date_val]
            if len(day_data) > 0:
                ec_ids = sorted(day_data['ec_id'].unique())
                loads = np.zeros((len(ec_ids), 24))
                pvs   = np.zeros((len(ec_ids), 24))
                for i, ec in enumerate(ec_ids):
                    ec_data = day_data[day_data['ec_id'] == ec].sort_values('HourUTC')
                    loads[i] = ec_data['Load'].values[:24]
                    pvs[i]   = ec_data['PV'].values[:24]
                return loads, pvs
    return None, None


def generate_synthetic_portfolio(base_profiles, n_ec, rng, prob_b, scale_mu, scale_sigma):
    """
    Generate a deterministic synthetic portfolio from base profiles.
    Each EC is a scaled version of either b-type or s-type.
    Scale factors: X ~ N(mu, sigma), excluding negatives.
    Returns: ec_types list, scale_factors array
    """
    ec_types = []
    scale_factors = []
    for _ in range(n_ec):
        # Type assignment
        ec_types.append('b' if rng.random() < prob_b else 's')
        # Scale factor (reject negatives)
        while True:
            s = rng.normal(scale_mu, scale_sigma)
            if s > 0:
                scale_factors.append(s)
                break
    return ec_types, np.array(scale_factors)


# ── Generate portfolio (deterministic via fixed seed) ─────────────────────
rng = np.random.default_rng(RNG_SEED)

# Try loading real profiles, fall back to synthetic base shapes
base_profiles = load_ec_base_profiles(EC_DIR)
if base_profiles is None:
    print('No base profiles found in data_ec/, using built-in synthetic shapes.')
    # Synthetic base shapes (representative residential patterns)
    hours = np.arange(24)
    base_profiles = {
        'b': {
            'load': 15 + 10 * np.sin(np.pi * (hours - 6) / 12) * (hours >= 6) * (hours <= 22) + \
                    8 * np.exp(-0.5 * ((hours - 18) / 2)**2),  # evening peak
            'pv':   np.maximum(0, 40 * np.sin(np.pi * (hours - 5) / 13)) * (hours >= 5) * (hours <= 18)
        },
        's': {
            'load': 12 + 8 * np.sin(np.pi * (hours - 7) / 11) * (hours >= 7) * (hours <= 21) + \
                    6 * np.exp(-0.5 * ((hours - 19) / 2)**2),
            'pv':   np.maximum(0, 50 * np.sin(np.pi * (hours - 5) / 13)) * (hours >= 5) * (hours <= 18)
        }
    }

ec_types, scale_factors = generate_synthetic_portfolio(
    base_profiles, N_EC, rng, PROB_B_TYPE, SCALE_MU, SCALE_SIGMA
)

print(f'Portfolio generated: {sum(t=="b" for t in ec_types)} b-type, '
      f'{sum(t=="s" for t in ec_types)} s-type')
print(f'Scale factors: {np.round(scale_factors, 3)}')

In [ ]:
def get_ec_profiles_for_day(date_val, df_day, ec_types, scale_factors, base_profiles, ec_dir):
    """
    Get Load and PV arrays for each EC for a given day.
    Shape: (N_EC, 24)
    
    Priority:
    1. Real per-EC daily data from ec_dir
    2. Aggregated columns in df_day, split by scale factors
    3. Synthetic from base profiles, scaled
    """
    n_ec = len(ec_types)
    
    # Attempt 1: real per-EC data
    loads, pvs = load_ec_daily_profiles(ec_dir, date_val)
    if loads is not None:
        return loads[:n_ec], pvs[:n_ec]
    
    # Attempt 2: synthetic from base profiles + seasonal PV scaling
    day_of_year = pd.Timestamp(date_val).day_of_year
    # Simple seasonal factor: peak in summer, low in winter (sinusoidal)
    seasonal_pv = 0.3 + 0.7 * np.sin(np.pi * (day_of_year - 80) / 365)  # peak ~June
    seasonal_pv = max(seasonal_pv, 0.05)
    
    loads_out = np.zeros((n_ec, 24))
    pvs_out   = np.zeros((n_ec, 24))
    
    for e in range(n_ec):
        t = ec_types[e]
        s = scale_factors[e]
        loads_out[e] = base_profiles[t]['load'] * s
        pvs_out[e]   = base_profiles[t]['pv'] * s * seasonal_pv
    
    return loads_out, pvs_out

## 4. MILP Model: Early Auction (§2.1)

Perfect forecasts: actual values are used as-is. No noise added.

In [ ]:
def build_early_model(loads, pvs, buy_price, sell_price,
                      fcrd_up_price, fcrd_down_price,
                      act_frac_up, act_frac_down,
                      y_acc_up, y_acc_down,
                      params):
    """
    Build the early auction MILP (§2.1 of thesis).
    
    All inputs are 1D arrays of length 24 (one day), except loads/pvs which
    are (N_EC, 24). These are PERFECT (actual) values — no forecasting error.
    
    Returns: Pyomo ConcreteModel
    """
    n_ec     = params['n_ec']
    b_max    = params['b_max']        # kW per EC
    s_max    = params['s_max']        # kWh per EC
    eta      = params['eta']
    soc_init = params['soc_init']     # kWh
    p_min    = params['p_min']        # kW min bid
    p_max    = params['p_max']        # kW max portfolio bid
    t_sus    = params['t_sustain']    # hours
    
    E = range(n_ec)
    T = range(T_MAX)
    
    m = pyo.ConcreteModel('EarlyAuction')
    m.E = pyo.Set(initialize=E)
    m.T = pyo.Set(initialize=T)
    
    # ── Decision variables (all >= 0) ─────────────────────────────────────
    m.p_im   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)   # import
    m.p_ex   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)   # export
    m.b_ch   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)   # charge
    m.b_dis  = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)   # discharge
    m.SOC    = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)   # state of charge
    m.p_up_res   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)  # FCR-D up reserved
    m.p_down_res = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)  # FCR-D down reserved
    m.p_up_act   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)  # FCR-D up activated
    m.p_down_act = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)  # FCR-D down activated
    
    # Net exchange (auxiliary, for notation parity with Cowley)
    m.p_net  = pyo.Var(m.E, m.T, within=pyo.Reals)
    
    # Binary: bid or not (portfolio level)
    m.z_up   = pyo.Var(m.T, within=pyo.Binary)
    m.z_down = pyo.Var(m.T, within=pyo.Binary)
    
    # ── Objective (m2a): minimise cost ════════════════════════════════════
    def obj_rule(m):
        return sum(
            buy_price[t] * (sum(m.p_im[e, t] for e in E) - sum(m.p_down_act[e, t] for e in E))
            - sell_price[t] * (sum(m.p_ex[e, t] for e in E) - sum(m.p_up_act[e, t] for e in E))
            - fcrd_up_price[t]   * sum(m.p_up_res[e, t] for e in E)
            - fcrd_down_price[t] * sum(m.p_down_res[e, t] for e in E)
            for t in T
        )
    m.objective = pyo.Objective(rule=obj_rule, sense=pyo.minimize)
    
    # ── Constraints ══════════════════════════════════════════════════════
    
    # (m1b) Net exchange definition: p_net = p_im - p_ex
    def net_exchange_rule(m, e, t):
        return m.p_net[e, t] - m.p_im[e, t] + m.p_ex[e, t] == 0
    m.c_net_exchange = pyo.Constraint(m.E, m.T, rule=net_exchange_rule)
    
    # (c2i) Power balance per EC:
    # p_net + PV - D - (b_ch + p_down_act) + (b_dis + p_up_act) = 0
    def power_balance_rule(m, e, t):
        return (m.p_net[e, t] + pvs[e, t] - loads[e, t]
                - (m.b_ch[e, t] + m.p_down_act[e, t])
                + (m.b_dis[e, t] + m.p_up_act[e, t]) == 0)
    m.c_power_balance = pyo.Constraint(m.E, m.T, rule=power_balance_rule)
    
    # (c3bup) Activation tracking: p_up_act = y_acc * y_act * p_up_res
    def act_up_rule(m, e, t):
        coeff = y_acc_up[t] * act_frac_up[t]
        return m.p_up_act[e, t] == coeff * m.p_up_res[e, t]
    m.c_act_up = pyo.Constraint(m.E, m.T, rule=act_up_rule)
    
    # (c3bdown)
    def act_down_rule(m, e, t):
        coeff = y_acc_down[t] * act_frac_down[t]
        return m.p_down_act[e, t] == coeff * m.p_down_res[e, t]
    m.c_act_down = pyo.Constraint(m.E, m.T, rule=act_down_rule)
    
    # (m2k) SOC initial: SOC(t=0) = 0.5*S_max + eta*(b_ch + p_down_act) - (1/eta)*(b_dis + p_up_act)
    def soc_init_rule(m, e):
        return (m.SOC[e, 0] == soc_init
                + eta * (m.b_ch[e, 0] + m.p_down_act[e, 0])
                - (1.0 / eta) * (m.b_dis[e, 0] + m.p_up_act[e, 0]))
    m.c_soc_init = pyo.Constraint(m.E, rule=soc_init_rule)
    
    # (m2j) SOC dynamics: t >= 1, t < T_MAX-1
    def soc_dynamics_rule(m, e, t):
        if t == 0:
            return pyo.Constraint.Skip
        if t == T_MAX - 1:
            return pyo.Constraint.Skip
        return (m.SOC[e, t] == m.SOC[e, t - 1]
                + eta * (m.b_ch[e, t] + m.p_down_act[e, t])
                - (1.0 / eta) * (m.b_dis[e, t] + m.p_up_act[e, t]))
    m.c_soc_dyn = pyo.Constraint(m.E, m.T, rule=soc_dynamics_rule)
    
    # (c1l) Terminal SOC = 0.5 * S_max
    def soc_terminal_rule(m, e):
        return m.SOC[e, T_MAX - 1] == soc_init
    m.c_soc_terminal = pyo.Constraint(m.E, rule=soc_terminal_rule)
    
    # (m13 up) SOC sustain for up: SOC >= y_acc * T_sustain / eta * p_up_res
    def soc_sustain_up_rule(m, e, t):
        coeff = y_acc_up[t] * t_sus / eta
        return m.SOC[e, t] >= coeff * m.p_up_res[e, t]
    m.c_soc_sustain_up = pyo.Constraint(m.E, m.T, rule=soc_sustain_up_rule)
    
    # (m13 down) Headroom sustain for down: S_max - SOC >= y_acc * T_sustain * eta * p_down_res
    def soc_sustain_down_rule(m, e, t):
        coeff = y_acc_down[t] * t_sus * eta
        return s_max - m.SOC[e, t] >= coeff * m.p_down_res[e, t]
    m.c_soc_sustain_down = pyo.Constraint(m.E, m.T, rule=soc_sustain_down_rule)
    
    # (c1m) Charge power limit: b_ch + p_down_res <= b_max
    def charge_limit_rule(m, e, t):
        return m.b_ch[e, t] + m.p_down_res[e, t] <= b_max
    m.c_charge_limit = pyo.Constraint(m.E, m.T, rule=charge_limit_rule)
    
    # (c3c) Discharge power limit: b_dis + p_up_res <= b_max
    def discharge_limit_rule(m, e, t):
        return m.b_dis[e, t] + m.p_up_res[e, t] <= b_max
    m.c_discharge_limit = pyo.Constraint(m.E, m.T, rule=discharge_limit_rule)
    
    # (c3d up) SOC >= p_up_res (must have energy to deliver)
    def soc_up_limit_rule(m, e, t):
        return m.SOC[e, t] >= m.p_up_res[e, t]
    m.c_soc_up_limit = pyo.Constraint(m.E, m.T, rule=soc_up_limit_rule)
    
    # (c3d down) Headroom >= p_down_res
    def soc_down_limit_rule(m, e, t):
        return s_max - m.SOC[e, t] >= m.p_down_res[e, t]
    m.c_soc_down_limit = pyo.Constraint(m.E, m.T, rule=soc_down_limit_rule)
    
    # SOC bounds
    def soc_upper_rule(m, e, t):
        return m.SOC[e, t] <= s_max
    m.c_soc_upper = pyo.Constraint(m.E, m.T, rule=soc_upper_rule)
    
    # ── Min bid size (m1up, m1down, m2): MILP constraints ─────────────────
    # p_min * z <= sum_e p_up_res <= P_max * z
    def min_bid_up_lo(m, t):
        return p_min * m.z_up[t] <= sum(m.p_up_res[e, t] for e in E)
    m.c_min_bid_up_lo = pyo.Constraint(m.T, rule=min_bid_up_lo)
    
    def min_bid_up_hi(m, t):
        return sum(m.p_up_res[e, t] for e in E) <= p_max * m.z_up[t]
    m.c_min_bid_up_hi = pyo.Constraint(m.T, rule=min_bid_up_hi)
    
    def min_bid_down_lo(m, t):
        return p_min * m.z_down[t] <= sum(m.p_down_res[e, t] for e in E)
    m.c_min_bid_down_lo = pyo.Constraint(m.T, rule=min_bid_down_lo)
    
    def min_bid_down_hi(m, t):
        return sum(m.p_down_res[e, t] for e in E) <= p_max * m.z_down[t]
    m.c_min_bid_down_hi = pyo.Constraint(m.T, rule=min_bid_down_hi)
    
    return m

## 5. MILP Model: Late Auction (§2.2)

Takes accepted early bids as given. Can top-up or cancel (buy back) per hour.

In [ ]:
def build_late_model(loads, pvs, buy_price, sell_price,
                     fcrd_up_early_price, fcrd_down_early_price,
                     fcrd_up_late_price, fcrd_down_late_price,
                     buyback_up_price, buyback_down_price,
                     act_frac_up, act_frac_down,
                     y_acc_up, y_acc_down,
                     early_up_accepted, early_down_accepted,
                     params):
    """
    Build the late auction MILP (§2.2 of thesis).
    
    early_up_accepted, early_down_accepted: arrays (N_EC, 24) of accepted
    FCR-D bids from the early auction (per EC). Under Assumption A1, these
    equal the early reservation variables.
    """
    n_ec     = params['n_ec']
    b_max    = params['b_max']
    s_max    = params['s_max']
    eta      = params['eta']
    soc_init = params['soc_init']
    p_min    = params['p_min']
    p_max    = params['p_max']
    t_sus    = params['t_sustain']
    
    E = range(n_ec)
    T = range(T_MAX)
    
    m = pyo.ConcreteModel('LateAuction')
    m.E = pyo.Set(initialize=E)
    m.T = pyo.Set(initialize=T)
    
    # ── Decision variables ────────────────────────────────────────────────
    m.p_im   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_ex   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.b_ch   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.b_dis  = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.SOC    = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_net  = pyo.Var(m.E, m.T, within=pyo.Reals)
    
    # Final total reservation (early accepted + late adjustments)
    m.p_up_res   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_down_res = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_up_act   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    m.p_down_act = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)
    
    # Late auction adjustments per EC
    m.a_up   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)  # top-up FCR-D up
    m.a_down = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)  # top-up FCR-D down
    m.c_up   = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)  # cancel FCR-D up
    m.c_down = pyo.Var(m.E, m.T, within=pyo.NonNegativeReals)  # cancel FCR-D down
    
    # Binaries
    m.w_up   = pyo.Var(m.T, within=pyo.Binary)  # 1 if topping up in hour t
    m.w_down = pyo.Var(m.T, within=pyo.Binary)
    m.z_up   = pyo.Var(m.T, within=pyo.Binary)  # 1 if final total reservation > 0
    m.z_down = pyo.Var(m.T, within=pyo.Binary)
    
    # ── Link final reservation to early + adjustments ─────────────────────
    def res_up_link(m, e, t):
        return m.p_up_res[e, t] == early_up_accepted[e, t] + m.a_up[e, t] - m.c_up[e, t]
    m.c_res_up_link = pyo.Constraint(m.E, m.T, rule=res_up_link)
    
    def res_down_link(m, e, t):
        return m.p_down_res[e, t] == early_down_accepted[e, t] + m.a_down[e, t] - m.c_down[e, t]
    m.c_res_down_link = pyo.Constraint(m.E, m.T, rule=res_down_link)
    
    # ── Objective (m2a late) ══════════════════════════════════════════════
    def obj_rule(m):
        return sum(
            # Import cost (adjusted for down activation)
            buy_price[t] * (sum(m.p_im[e, t] for e in E) - sum(m.p_down_act[e, t] for e in E))
            # Export revenue (adjusted for up activation)
            - sell_price[t] * (sum(m.p_ex[e, t] for e in E) - sum(m.p_up_act[e, t] for e in E))
            # Early reservation revenue (on accepted bids, fixed)
            - fcrd_up_early_price[t]   * sum(early_up_accepted[e, t] for e in E)
            - fcrd_down_early_price[t] * sum(early_down_accepted[e, t] for e in E)
            # Buyback cost (cancellation)
            + buyback_up_price[t]   * sum(m.c_up[e, t] for e in E)
            + buyback_down_price[t] * sum(m.c_down[e, t] for e in E)
            # Late top-up revenue
            - fcrd_up_late_price[t]   * sum(m.a_up[e, t] for e in E)
            - fcrd_down_late_price[t] * sum(m.a_down[e, t] for e in E)
            for t in T
        )
    m.objective = pyo.Objective(rule=obj_rule, sense=pyo.minimize)
    
    # ── Same base constraints as early ────────────────────────────────────
    def net_exchange_rule(m, e, t):
        return m.p_net[e, t] - m.p_im[e, t] + m.p_ex[e, t] == 0
    m.c_net_exchange = pyo.Constraint(m.E, m.T, rule=net_exchange_rule)
    
    def power_balance_rule(m, e, t):
        return (m.p_net[e, t] + pvs[e, t] - loads[e, t]
                - (m.b_ch[e, t] + m.p_down_act[e, t])
                + (m.b_dis[e, t] + m.p_up_act[e, t]) == 0)
    m.c_power_balance = pyo.Constraint(m.E, m.T, rule=power_balance_rule)
    
    def act_up_rule(m, e, t):
        return m.p_up_act[e, t] == y_acc_up[t] * act_frac_up[t] * m.p_up_res[e, t]
    m.c_act_up = pyo.Constraint(m.E, m.T, rule=act_up_rule)
    
    def act_down_rule(m, e, t):
        return m.p_down_act[e, t] == y_acc_down[t] * act_frac_down[t] * m.p_down_res[e, t]
    m.c_act_down = pyo.Constraint(m.E, m.T, rule=act_down_rule)
    
    def soc_init_rule(m, e):
        return (m.SOC[e, 0] == soc_init
                + eta * (m.b_ch[e, 0] + m.p_down_act[e, 0])
                - (1.0 / eta) * (m.b_dis[e, 0] + m.p_up_act[e, 0]))
    m.c_soc_init = pyo.Constraint(m.E, rule=soc_init_rule)
    
    def soc_dynamics_rule(m, e, t):
        if t == 0 or t == T_MAX - 1:
            return pyo.Constraint.Skip
        return (m.SOC[e, t] == m.SOC[e, t - 1]
                + eta * (m.b_ch[e, t] + m.p_down_act[e, t])
                - (1.0 / eta) * (m.b_dis[e, t] + m.p_up_act[e, t]))
    m.c_soc_dyn = pyo.Constraint(m.E, m.T, rule=soc_dynamics_rule)
    
    def soc_terminal_rule(m, e):
        return m.SOC[e, T_MAX - 1] == soc_init
    m.c_soc_terminal = pyo.Constraint(m.E, rule=soc_terminal_rule)
    
    def soc_sustain_up_rule(m, e, t):
        return m.SOC[e, t] >= y_acc_up[t] * t_sus / eta * m.p_up_res[e, t]
    m.c_soc_sustain_up = pyo.Constraint(m.E, m.T, rule=soc_sustain_up_rule)
    
    def soc_sustain_down_rule(m, e, t):
        return s_max - m.SOC[e, t] >= y_acc_down[t] * t_sus * eta * m.p_down_res[e, t]
    m.c_soc_sustain_down = pyo.Constraint(m.E, m.T, rule=soc_sustain_down_rule)
    
    def charge_limit_rule(m, e, t):
        return m.b_ch[e, t] + m.p_down_res[e, t] <= b_max
    m.c_charge_limit = pyo.Constraint(m.E, m.T, rule=charge_limit_rule)
    
    def discharge_limit_rule(m, e, t):
        return m.b_dis[e, t] + m.p_up_res[e, t] <= b_max
    m.c_discharge_limit = pyo.Constraint(m.E, m.T, rule=discharge_limit_rule)
    
    def soc_up_limit_rule(m, e, t):
        return m.SOC[e, t] >= m.p_up_res[e, t]
    m.c_soc_up_limit = pyo.Constraint(m.E, m.T, rule=soc_up_limit_rule)
    
    def soc_down_limit_rule(m, e, t):
        return s_max - m.SOC[e, t] >= m.p_down_res[e, t]
    m.c_soc_down_limit = pyo.Constraint(m.E, m.T, rule=soc_down_limit_rule)
    
    def soc_upper_rule(m, e, t):
        return m.SOC[e, t] <= s_max
    m.c_soc_upper = pyo.Constraint(m.E, m.T, rule=soc_upper_rule)
    
    # ── Min bid for top-ups ───────────────────────────────────────────────
    def topup_up_lo(m, t):
        return p_min * m.w_up[t] <= sum(m.a_up[e, t] for e in E)
    m.c_topup_up_lo = pyo.Constraint(m.T, rule=topup_up_lo)
    
    def topup_up_hi(m, t):
        return sum(m.a_up[e, t] for e in E) <= p_max * m.w_up[t]
    m.c_topup_up_hi = pyo.Constraint(m.T, rule=topup_up_hi)
    
    def topup_down_lo(m, t):
        return p_min * m.w_down[t] <= sum(m.a_down[e, t] for e in E)
    m.c_topup_down_lo = pyo.Constraint(m.T, rule=topup_down_lo)
    
    def topup_down_hi(m, t):
        return sum(m.a_down[e, t] for e in E) <= p_max * m.w_down[t]
    m.c_topup_down_hi = pyo.Constraint(m.T, rule=topup_down_hi)
    
    # ── Min bid for final total reservation ──────────────────────────────
    def final_up_lo(m, t):
        return p_min * m.z_up[t] <= sum(m.p_up_res[e, t] for e in E)
    m.c_final_up_lo = pyo.Constraint(m.T, rule=final_up_lo)
    
    def final_up_hi(m, t):
        return sum(m.p_up_res[e, t] for e in E) <= p_max * m.z_up[t]
    m.c_final_up_hi = pyo.Constraint(m.T, rule=final_up_hi)
    
    def final_down_lo(m, t):
        return p_min * m.z_down[t] <= sum(m.p_down_res[e, t] for e in E)
    m.c_final_down_lo = pyo.Constraint(m.T, rule=final_down_lo)
    
    def final_down_hi(m, t):
        return sum(m.p_down_res[e, t] for e in E) <= p_max * m.z_down[t]
    m.c_final_down_hi = pyo.Constraint(m.T, rule=final_down_hi)
    
    # ── Cannot cancel and top-up simultaneously ──────────────────────────
    # c_up <= early_accepted * (1 - w_up)
    def no_simul_up(m, e, t):
        return m.c_up[e, t] <= early_up_accepted[e, t] * (1 - m.w_up[t])
    m.c_no_simul_up = pyo.Constraint(m.E, m.T, rule=no_simul_up)
    
    def no_simul_down(m, e, t):
        return m.c_down[e, t] <= early_down_accepted[e, t] * (1 - m.w_down[t])
    m.c_no_simul_down = pyo.Constraint(m.E, m.T, rule=no_simul_down)
    
    return m

## 6. Solver & extraction helpers

In [ ]:
solver = Highs()
solver.config.time_limit = 120  # seconds per solve
solver.config.mip_rel_gap = 1e-4


def solve_model(model):
    """Solve a Pyomo model with HiGHS. Returns results object."""
    results = solver.solve(model)
    if results.termination_condition not in (
        pyo.TerminationCondition.optimal,
        pyo.TerminationCondition.feasible,
    ):
        raise RuntimeError(f'Solver failed: {results.termination_condition}')
    return results


def extract_var_2d(var, n_ec, t_max=T_MAX):
    """Extract a 2D Pyomo variable (e, t) → numpy array (n_ec, t_max)."""
    arr = np.zeros((n_ec, t_max))
    for e in range(n_ec):
        for t in range(t_max):
            arr[e, t] = pyo.value(var[e, t])
    return arr


def extract_var_1d(var, t_max=T_MAX):
    """Extract a 1D Pyomo variable (t) → numpy array."""
    return np.array([pyo.value(var[t]) for t in range(t_max)])


print('Solver ready')

## 7. Daily solve function (Early → Late, perfect forecasts)

In [ ]:
def solve_day(date_val, df_day, loads, pvs, params, also_base=True):
    """
    Solve the sequential auction for one day with PERFECT forecasts.
    
    Returns a dict with all results needed for aggregation.
    If also_base=True, also solves a base case (no FCR-D) for comparison.
    """
    n_ec = params['n_ec']
    
    # ── Extract day's market data (perfect = actual) ──────────────────────
    buy_price  = df_day['buy_price'].values
    sell_price = df_day['sell_price'].values
    
    fcrd_up_early   = df_day[COL_FCRD_UP_EARLY].values
    fcrd_down_early = df_day[COL_FCRD_DOWN_EARLY].values
    fcrd_up_late    = df_day[COL_FCRD_UP_LATE].values
    fcrd_down_late  = df_day[COL_FCRD_DOWN_LATE].values
    
    act_frac_up   = df_day[COL_ACT_FRAC_UP].values
    act_frac_down = df_day[COL_ACT_FRAC_DOWN].values
    
    y_acc_up   = df_day['y_acc_up'].values
    y_acc_down = df_day['y_acc_down'].values
    
    buyback_up   = df_day['buyback_up'].values
    buyback_down = df_day['buyback_down'].values
    
    # ── Step 1: Solve EARLY auction ───────────────────────────────────────
    m_early = build_early_model(
        loads, pvs, buy_price, sell_price,
        fcrd_up_early, fcrd_down_early,
        act_frac_up, act_frac_down,
        y_acc_up, y_acc_down,
        params
    )
    solve_model(m_early)
    
    # Extract early results
    early_up_res   = extract_var_2d(m_early.p_up_res, n_ec)
    early_down_res = extract_var_2d(m_early.p_down_res, n_ec)
    early_soc      = extract_var_2d(m_early.SOC, n_ec)
    early_b_ch     = extract_var_2d(m_early.b_ch, n_ec)
    early_b_dis    = extract_var_2d(m_early.b_dis, n_ec)
    early_p_im     = extract_var_2d(m_early.p_im, n_ec)
    early_p_ex     = extract_var_2d(m_early.p_ex, n_ec)
    
    # Accepted bids = reserved bids (Assumption A1: bid at price 0, all accepted)
    early_up_accepted   = early_up_res.copy()
    early_down_accepted = early_down_res.copy()
    
    # ── Step 2: Solve LATE auction ────────────────────────────────────────
    m_late = build_late_model(
        loads, pvs, buy_price, sell_price,
        fcrd_up_early, fcrd_down_early,
        fcrd_up_late, fcrd_down_late,
        buyback_up, buyback_down,
        act_frac_up, act_frac_down,
        y_acc_up, y_acc_down,
        early_up_accepted, early_down_accepted,
        params
    )
    solve_model(m_late)
    
    # Extract late results
    final_up_res   = extract_var_2d(m_late.p_up_res, n_ec)
    final_down_res = extract_var_2d(m_late.p_down_res, n_ec)
    final_soc      = extract_var_2d(m_late.SOC, n_ec)
    final_b_ch     = extract_var_2d(m_late.b_ch, n_ec)
    final_b_dis    = extract_var_2d(m_late.b_dis, n_ec)
    final_p_im     = extract_var_2d(m_late.p_im, n_ec)
    final_p_ex     = extract_var_2d(m_late.p_ex, n_ec)
    final_up_act   = extract_var_2d(m_late.p_up_act, n_ec)
    final_down_act = extract_var_2d(m_late.p_down_act, n_ec)
    topup_up       = extract_var_2d(m_late.a_up, n_ec)
    topup_down     = extract_var_2d(m_late.a_down, n_ec)
    cancel_up      = extract_var_2d(m_late.c_up, n_ec)
    cancel_down    = extract_var_2d(m_late.c_down, n_ec)
    
    # ── Revenue breakdown (aggregated across ECs) ─────────────────────────
    agg_p_im     = final_p_im.sum(axis=0)      # (24,)
    agg_p_ex     = final_p_ex.sum(axis=0)
    agg_up_res   = final_up_res.sum(axis=0)
    agg_down_res = final_down_res.sum(axis=0)
    agg_up_act   = final_up_act.sum(axis=0)
    agg_down_act = final_down_act.sum(axis=0)
    agg_topup_up   = topup_up.sum(axis=0)
    agg_topup_down = topup_down.sum(axis=0)
    agg_cancel_up  = cancel_up.sum(axis=0)
    agg_cancel_down = cancel_down.sum(axis=0)
    agg_early_up   = early_up_accepted.sum(axis=0)
    agg_early_down = early_down_accepted.sum(axis=0)
    
    # Energy cost
    import_cost = (buy_price * (agg_p_im - agg_down_act)).sum()
    export_rev  = (sell_price * (agg_p_ex - agg_up_act)).sum()
    net_energy_cost = import_cost - export_rev
    
    # FCR-D revenue
    fcrd_up_early_rev   = (fcrd_up_early   * agg_early_up).sum()
    fcrd_down_early_rev = (fcrd_down_early * agg_early_down).sum()
    fcrd_up_late_rev    = (fcrd_up_late   * agg_topup_up).sum()
    fcrd_down_late_rev  = (fcrd_down_late * agg_topup_down).sum()
    buyback_up_cost     = (buyback_up * agg_cancel_up).sum()
    buyback_down_cost   = (buyback_down * agg_cancel_down).sum()
    
    fcrd_up_net   = fcrd_up_early_rev   + fcrd_up_late_rev   - buyback_up_cost
    fcrd_down_net = fcrd_down_early_rev + fcrd_down_late_rev - buyback_down_cost
    
    result = {
        'date': date_val,
        'net_energy_cost': net_energy_cost,
        'import_cost': import_cost,
        'export_rev': export_rev,
        'fcrd_up_early_rev': fcrd_up_early_rev,
        'fcrd_down_early_rev': fcrd_down_early_rev,
        'fcrd_up_late_rev': fcrd_up_late_rev,
        'fcrd_down_late_rev': fcrd_down_late_rev,
        'buyback_up_cost': buyback_up_cost,
        'buyback_down_cost': buyback_down_cost,
        'fcrd_up_net': fcrd_up_net,
        'fcrd_down_net': fcrd_down_net,
        'total_fcrd_rev': fcrd_up_net + fcrd_down_net,
        # Hourly vectors for plotting
        'early_up_res': agg_early_up,
        'early_down_res': agg_early_down,
        'final_up_res': agg_up_res,
        'final_down_res': agg_down_res,
        'topup_up': agg_topup_up,
        'topup_down': agg_topup_down,
        'cancel_up': agg_cancel_up,
        'cancel_down': agg_cancel_down,
        'soc': final_soc.sum(axis=0),  # aggregated SOC
        'b_ch': final_b_ch.sum(axis=0),
        'b_dis': final_b_dis.sum(axis=0),
        'p_im': agg_p_im,
        'p_ex': agg_p_ex,
    }
    
    # ── Base case: no FCR-D (battery only for self-consumption + arbitrage)
    if also_base:
        m_base = build_early_model(
            loads, pvs, buy_price, sell_price,
            np.zeros(T_MAX), np.zeros(T_MAX),  # zero FCR-D prices → no bidding
            act_frac_up, act_frac_down,
            np.zeros(T_MAX), np.zeros(T_MAX),
            params
        )
        solve_model(m_base)
        base_p_im = extract_var_2d(m_base.p_im, n_ec).sum(axis=0)
        base_p_ex = extract_var_2d(m_base.p_ex, n_ec).sum(axis=0)
        base_import_cost = (buy_price * base_p_im).sum()
        base_export_rev  = (sell_price * base_p_ex).sum()
        result['base_energy_cost'] = base_import_cost - base_export_rev
        
        # No-battery baseline
        agg_load = loads.sum(axis=0)
        agg_pv   = pvs.sum(axis=0)
        net_load = agg_load - agg_pv
        no_batt_import = np.maximum(net_load, 0)
        no_batt_export = np.maximum(-net_load, 0)
        result['no_battery_cost'] = (buy_price * no_batt_import - sell_price * no_batt_export).sum()
    
    return result


print('solve_day() defined')

## 8. Single day example: May 10, 2025

In [ ]:
from datetime import date as dt_date

example_date = dt_date(2025, 5, 10)

# Get data for this day
df_day = df_all[df_all['date'] == example_date].sort_values(COL_HOUR).reset_index(drop=True)
assert len(df_day) == 24, f'Expected 24 hours, got {len(df_day)} for {example_date}'

# Get EC profiles
loads_day, pvs_day = get_ec_profiles_for_day(
    example_date, df_day, ec_types, scale_factors, base_profiles, EC_DIR
)

# Solve
params = {
    'n_ec': N_EC, 'b_max': MAX_POWER_EC, 's_max': CAPACITY_EC,
    'eta': ETA, 'soc_init': SOC_INIT_FRAC * CAPACITY_EC,
    'p_min': P_MIN_BID, 'p_max': P_MAX_PORTFOLIO, 't_sustain': T_SUSTAIN,
}

res = solve_day(example_date, df_day, loads_day, pvs_day, params)

print(f'\n=== {example_date} results ===')
print(f'Net energy cost (with FCR-D):   {res["net_energy_cost"]:>10.1f} DKK')
print(f'Net energy cost (base, no FCRD):{res["base_energy_cost"]:>10.1f} DKK')
print(f'No battery cost:                {res["no_battery_cost"]:>10.1f} DKK')
print(f'FCR-D Up net revenue:           {res["fcrd_up_net"]:>10.1f} DKK')
print(f'FCR-D Down net revenue:         {res["fcrd_down_net"]:>10.1f} DKK')
print(f'Total FCR-D revenue:            {res["total_fcrd_rev"]:>10.1f} DKK')

In [ ]:
# ── Figure 2: Production, Consumption & Prices / SOC ─────────────────────
hours = np.arange(24)
buy_p  = df_day['buy_price'].values
sell_p = df_day['sell_price'].values

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.suptitle(f'Production, Consumption & Prices — {example_date}', fontsize=13)

ax = axes[0]
agg_pv   = pvs_day.sum(axis=0)
agg_load = loads_day.sum(axis=0)
net_batt = res['b_ch'] - res['b_dis']
ax.bar(hours, agg_pv, color='gold', alpha=0.7, label='PV production')
ax.bar(hours, -agg_load, color='steelblue', alpha=0.7, label='Demand (−)')
ax.step(hours, net_batt, color='green', linewidth=1.5, where='mid', label='Net battery (ch−dis)')
ax.set_ylabel('kW')
ax.legend(loc='upper left', fontsize=8)
ax2 = ax.twinx()
ax2.plot(hours, buy_p, 'r--', linewidth=1, label='Buy price')
ax2.plot(hours, sell_p, 'b--', linewidth=1, label='Sell price')
ax2.set_ylabel('Price (DKK/kWh)')
ax2.legend(loc='upper right', fontsize=8)

ax = axes[1]
ax.plot(hours, res['soc'], 'r-o', markersize=4, label='SOC')
ax.axhline(S_MAX_PORTFOLIO, color='grey', linestyle=':', label=f'Cap ({S_MAX_PORTFOLIO} kWh)')
ax.fill_between(hours, 0, S_MAX_PORTFOLIO, alpha=0.05, color='grey')
ax.set_ylabel('SOC (kWh)')
ax.set_xlabel('Hour (UTC)')
ax.legend(loc='upper left', fontsize=8)
ax3 = ax.twinx()
ax3.plot(hours, buy_p, 'r--', linewidth=1, label='Buy price')
ax3.plot(hours, sell_p, 'b--', linewidth=1, label='Sell price')
ax3.set_ylabel('Price (DKK/kWh)')
ax3.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'arbitrage.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 4: Sequential auction reservations ────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle(f'Sequential Auction — {example_date}', fontsize=13)

# Early auction reservations
ax = axes[0]
ax.set_title('Early auction reservations (perfect forecasts)')
w = 0.35
ax.bar(hours - w/2, res['early_up_res'], w, color='salmon', label='Early Up reserved')
ax.bar(hours + w/2, res['early_down_res'], w, color='mediumaquamarine', label='Early Down reserved')
ax.axhline(P_MIN_BID, color='grey', linestyle=':', label=f'Min bid ({P_MIN_BID} kW)')
ax.set_ylabel('kW')
ax.legend(fontsize=8)

# Final reservations after late
ax = axes[1]
ax.set_title('Final reservations after late auction (perfect forecasts)')
ax.bar(hours - w/2, res['final_up_res'], w, color='salmon', label='Final Up reserved')
ax.bar(hours + w/2, res['final_down_res'], w, color='mediumaquamarine', label='Final Down reserved')
ax.set_ylabel('kW')
ax.legend(fontsize=8)

# Late auction adjustments
ax = axes[2]
ax.set_title('Late auction adjustments (cancellations below 0, top-ups above)')
ax.bar(hours - w/2, res['topup_up'], w, color='salmon', alpha=0.7, label='Top-up Up')
ax.bar(hours - w/2, -res['cancel_up'], w, color='darkred', alpha=0.5, label='Cancel Up', hatch='//')
ax.bar(hours + w/2, res['topup_down'], w, color='mediumaquamarine', alpha=0.7, label='Top-up Down')
ax.bar(hours + w/2, -res['cancel_down'], w, color='teal', alpha=0.5, label='Cancel Down', hatch='//')
ax.set_ylabel('kW')
ax.set_xlabel('Hour (UTC)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'reservations1.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 5: SOC & Feasibility Corridor ─────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(10, 8))
fig.suptitle(f'SOC & Feasibility Corridor — {example_date}', fontsize=13)

# Compute forbidden zones from final reservations
y_acc_up_day   = df_day['y_acc_up'].values
y_acc_down_day = df_day['y_acc_down'].values
forbidden_up   = y_acc_up_day * T_SUSTAIN / ETA * res['final_up_res']   # min SOC for up
forbidden_down = y_acc_down_day * T_SUSTAIN * ETA * res['final_down_res']  # headroom for down

ax = axes[0]
ax.set_title('SOC & Feasibility Corridor')
ax.fill_between(hours, 0, forbidden_up, alpha=0.3, color='salmon', label='Forbidden (Up Sustain)')
ax.fill_between(hours, S_MAX_PORTFOLIO - forbidden_down, S_MAX_PORTFOLIO,
                alpha=0.3, color='mediumaquamarine', label='Forbidden (Dn Sustain)')
ax.plot(hours, res['soc'], 'r-o', markersize=5, label='SOC')
ax.axhline(S_MAX_PORTFOLIO, color='grey', linestyle=':')
ax.set_ylabel('SOC (kWh)')
ax.legend(fontsize=8)

# Late auction adjustments
ax = axes[1]
ax.set_title('Late Auction Adjustments')
w = 0.35
ax.bar(hours - w/2, res['topup_up'], w, color='salmon', label='Top-up Up')
ax.bar(hours + w/2, res['topup_down'], w, color='mediumaquamarine', label='Top-up Down')
ax.bar(hours - w/2, -res['cancel_up'], w, color='darkred', alpha=0.6, label='Cancel Up', hatch='//')
ax.bar(hours + w/2, -res['cancel_down'], w, color='teal', alpha=0.6, label='Cancel Down', hatch='//')
ax.set_ylabel('Adjustment (kW)')
ax.set_xlabel('Hour (UTC)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'dashboard2.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Full year back-test (2025, perfect forecasts)

Run the sequential early + late auction MILP for every day with actual data.
This produces the **upper bound** on revenue.

In [ ]:
# Get all unique dates in dataset
all_dates = sorted(df_all['date'].unique())
print(f'Total days to solve: {len(all_dates)}')

# Filter to complete 24-hour days only
day_counts = df_all.groupby('date').size()
complete_dates = [d for d in all_dates if day_counts[d] == 24]
print(f'Complete 24h days: {len(complete_dates)}')

In [ ]:
# ── Run the full year ─────────────────────────────────────────────────────
results_list = []
failed_days  = []

params = {
    'n_ec': N_EC, 'b_max': MAX_POWER_EC, 's_max': CAPACITY_EC,
    'eta': ETA, 'soc_init': SOC_INIT_FRAC * CAPACITY_EC,
    'p_min': P_MIN_BID, 'p_max': P_MAX_PORTFOLIO, 't_sustain': T_SUSTAIN,
}

t_start = time.time()
for i, date_val in enumerate(complete_dates):
    try:
        df_day = df_all[df_all['date'] == date_val].sort_values(COL_HOUR).reset_index(drop=True)
        
        loads_day, pvs_day = get_ec_profiles_for_day(
            date_val, df_day, ec_types, scale_factors, base_profiles, EC_DIR
        )
        
        res = solve_day(date_val, df_day, loads_day, pvs_day, params, also_base=True)
        results_list.append(res)
        
        if (i + 1) % 30 == 0 or (i + 1) == len(complete_dates):
            elapsed = time.time() - t_start
            print(f'  [{i+1:>3}/{len(complete_dates)}] {date_val}  '
                  f'FCRD={res["total_fcrd_rev"]:>7.0f} DKK  '
                  f'elapsed={elapsed:.0f}s')
    except Exception as ex:
        failed_days.append((date_val, str(ex)))
        print(f'  FAILED {date_val}: {ex}')

elapsed = time.time() - t_start
print(f'\nDone. {len(results_list)} days solved in {elapsed:.0f}s, {len(failed_days)} failed.')

In [ ]:
# ── Build results DataFrame ──────────────────────────────────────────────
scalar_cols = ['date', 'net_energy_cost', 'import_cost', 'export_rev',
               'fcrd_up_early_rev', 'fcrd_down_early_rev',
               'fcrd_up_late_rev', 'fcrd_down_late_rev',
               'buyback_up_cost', 'buyback_down_cost',
               'fcrd_up_net', 'fcrd_down_net', 'total_fcrd_rev',
               'base_energy_cost', 'no_battery_cost']

df_results = pd.DataFrame([{k: r[k] for k in scalar_cols if k in r} for r in results_list])
df_results['date'] = pd.to_datetime(df_results['date'])
df_results['month'] = df_results['date'].dt.month

# Battery arbitrage saving = no_battery_cost - base_energy_cost
df_results['batt_arb_saving'] = df_results['no_battery_cost'] - df_results['base_energy_cost']

# Arb improvement from FCR-D = base_energy_cost - net_energy_cost (negative means FCR-D hurts arb)
df_results['arb_improvement_fcrd'] = df_results['base_energy_cost'] - df_results['net_energy_cost']

# Total value = arb saving + arb improvement + FCRD revenue
df_results['total_value'] = (df_results['batt_arb_saving']
                             + df_results['arb_improvement_fcrd']
                             + df_results['total_fcrd_rev'])

print(df_results[['date', 'batt_arb_saving', 'arb_improvement_fcrd',
                  'fcrd_up_net', 'fcrd_down_net', 'total_value']].describe())

## 10. Yearly revenue summary (Table 3)

In [ ]:
# ── Table 3: Yearly revenue stream summary ───────────────────────────────
total_batt_arb  = df_results['batt_arb_saving'].sum()
total_arb_impr  = df_results['arb_improvement_fcrd'].sum()
total_up_net    = df_results['fcrd_up_net'].sum()
total_down_net  = df_results['fcrd_down_net'].sum()
total_val       = df_results['total_value'].sum()

fcrd_total = total_up_net + total_down_net

print('=' * 55)
print(f'{"Revenue stream":<35} {"Value (TDKK)":>15}')
print('-' * 55)
print(f'{"Battery arbitrage saving":<35} {total_batt_arb/1000:>15.1f}')
print(f'{"Arb improvement from FCR-D":<35} {total_arb_impr/1000:>15.1f}')
print(f'{"FCR-D Up net revenue":<35} {total_up_net/1000:>15.1f}')
print(f'{"FCR-D Down net revenue":<35} {total_down_net/1000:>15.1f}')
print('-' * 55)
print(f'{"Total value":<35} {total_val/1000:>15.1f}')
print(f'{"FCR-D share of total":<35} {fcrd_total/total_val*100:>14.1f}%')
print(f'{"Battery arb share of total":<35} {total_batt_arb/total_val*100:>14.1f}%')
print('=' * 55)

## 11. Yearly plots (Figures 6 & 7)

In [ ]:
# ── Figure 6: Monthly cost / revenue decomposition ───────────────────────
monthly = df_results.groupby('month').agg({
    'no_battery_cost':    'sum',
    'base_energy_cost':   'sum',
    'net_energy_cost':    'sum',
    'fcrd_up_early_rev':  'sum',
    'fcrd_down_early_rev':'sum',
    'fcrd_up_late_rev':   'sum',
    'fcrd_down_late_rev': 'sum',
    'buyback_up_cost':    'sum',
    'buyback_down_cost':  'sum',
    'total_fcrd_rev':     'sum',
    'batt_arb_saving':    'sum',
    'arb_improvement_fcrd': 'sum',
    'fcrd_up_net':        'sum',
    'fcrd_down_net':      'sum',
    'total_value':        'sum',
}).reset_index()

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
months = monthly['month'].values

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(f'Sequential FCR-D — {N_EC} ECs ({P_MAX_PORTFOLIO} kW / {S_MAX_PORTFOLIO} kWh)',
             fontsize=13)

# Top-left: Monthly arbitrage cost by strategy
ax = axes[0, 0]
ax.set_title('Monthly arbitrage cost by strategy')
w = 0.25
ax.bar(months - w, monthly['no_battery_cost'] / 1000, w, label='No battery', color='lightgrey')
ax.bar(months,     monthly['base_energy_cost'] / 1000, w, label='Base', color='steelblue')
ax.bar(months + w, monthly['net_energy_cost'] / 1000, w, label='Early+Late', color='darkblue')
ax.set_ylabel('Net electricity cost (TDKK)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels, fontsize=8)
ax.legend(fontsize=8)

# Top-right: Monthly FCR-D revenue decomposition
ax = axes[0, 1]
ax.set_title('Monthly FCR-D revenue decomposition (Early+Late)')
early_rev = (monthly['fcrd_up_early_rev'] + monthly['fcrd_down_early_rev']) / 1000
topup_rev = (monthly['fcrd_up_late_rev'] + monthly['fcrd_down_late_rev']) / 1000
cancel_cost = (monthly['buyback_up_cost'] + monthly['buyback_down_cost']) / 1000
ax.bar(months, early_rev, label='Early revenue', color='salmon')
ax.bar(months, topup_rev, bottom=early_rev, label='Top-up revenue', color='mediumaquamarine')
ax.bar(months, -cancel_cost, label='Cancellation cost', color='darkred', alpha=0.6)
ax.set_ylabel('Capacity revenue (TDKK)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels, fontsize=8)
ax.legend(fontsize=8)

# Bottom-left: Adjustment volumes
ax = axes[1, 0]
ax.set_title('Monthly late-auction adjustment volumes')
# We need to aggregate hourly vectors — use cancel/topup sums from daily
cancel_vol = df_results.groupby('month').apply(
    lambda g: g['buyback_up_cost'].sum() + g['buyback_down_cost'].sum()).values / 1000
topup_vol = df_results.groupby('month').apply(
    lambda g: g['fcrd_up_late_rev'].sum() + g['fcrd_down_late_rev'].sum()).values / 1000
ax.bar(months - 0.15, topup_vol, 0.3, label='Top-up (TDKK)', color='mediumaquamarine')
ax.bar(months + 0.15, cancel_vol, 0.3, label='Cancelled (TDKK)', color='darkred', alpha=0.6)
ax.set_ylabel('Value (TDKK)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels, fontsize=8)
ax.legend(fontsize=8)

# Bottom-right: Monthly value of late auction adjustments
ax = axes[1, 1]
ax.set_title('Monthly value of late auction adjustments')
late_net = (monthly['fcrd_up_late_rev'] + monthly['fcrd_down_late_rev']
            - monthly['buyback_up_cost'] - monthly['buyback_down_cost']) / 1000
ax.bar(months, late_net, color='teal')
ax.set_ylabel('Uplift from late auction (TDKK)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels, fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'YearlyRun.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 7: Monthly revenue streams breakdown ──────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle(f'Monthly revenue streams — {N_EC} ECs ({P_MAX_PORTFOLIO} kW / {S_MAX_PORTFOLIO} kWh)',
             fontsize=13)
ax.set_title('Revenue stream decomposition by month (Early+Late strategy)')

w = 0.18
ax.bar(months - 1.5*w, monthly['batt_arb_saving'] / 1000, w,
       label='Battery arbitrage saving', color='steelblue')
ax.bar(months - 0.5*w, monthly['arb_improvement_fcrd'] / 1000, w,
       label='Arb improvement from FCR-D', color='lightblue')
ax.bar(months + 0.5*w, monthly['fcrd_up_net'] / 1000, w,
       label='FCR-D Up net revenue', color='salmon')
ax.bar(months + 1.5*w, monthly['fcrd_down_net'] / 1000, w,
       label='FCR-D Down net revenue', color='mediumaquamarine')

ax.plot(months, monthly['total_value'] / 1000, 'k-o', markersize=5, label='Total value')

ax.set_ylabel('Value (TDKK)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.legend(fontsize=9)
ax.axhline(0, color='grey', linewidth=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'YearlyRevBreakdown.png'), dpi=150, bbox_inches='tight')
plt.show()

## 12. Save results

In [ ]:
# Save daily results to CSV for use in 02_Sensitivity and further analysis
output_csv = os.path.join(DATA_OUT_DIR, 'backtest_upper_bound_2025.csv')
df_results.to_csv(output_csv, index=False)
print(f'Results saved to {output_csv}')
print(f'\nYearly total value: {df_results["total_value"].sum()/1000:.1f} TDKK')
print(f'FCR-D share: {(df_results["fcrd_up_net"].sum() + df_results["fcrd_down_net"].sum()) / df_results["total_value"].sum() * 100:.1f}%')